# UC-06 - Notebook d'exploration (store_sales et store_returns)

Exploration PySpark des deux tables de faits retenues, lues dans la zone Raw du Data Lake HDFS (jeu initial + rafraichissement). Les memes controles sont rejoues par le script `03_exploration.py`.

## 1. Environnement et session Spark

In [ ]:
import sys, platform
sys.path.insert(0, '/scripts')
from commun import *
from nettoyage import charger_type, echecs_conversion
spark = session('UC06 - notebook exploration')
print('Python :', platform.python_version(), '| Spark :', spark.version, '| master :', spark.sparkContext.master)
print('Heure  :', heure())

## 2. store_sales : lecture, schema, echantillon, volume

In [ ]:
ss = charger_type(spark, 'store_sales')
ss.printSchema()
ss.show(5, truncate=30)
print('Lignes :', ss.count())

## 3. Valeurs manquantes (store_sales)

In [ ]:
n = ss.count()
manq = ss.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in noms('store_sales')]).first()
for c in noms('store_sales'):
    if manq[c]:
        print(f'{c:20s} {manq[c]:>8d} ({100 * manq[c] / n:.2f} %)')

## 4. Doublons et cle candidate (store_sales)

In [ ]:
cle = CLES['store_sales']
print('Lignes strictement identiques :', n - ss.dropDuplicates(noms('store_sales')).count())
print('Repetitions de la cle', cle, ':', n - ss.dropDuplicates(cle).count())
ss.groupBy(*cle).count().filter('count > 1').orderBy(F.desc('count')).show(5)

## 5. Valeurs atypiques (store_sales)

Une valeur negative ou un profit negatif n'est pas une erreur par principe : on les compte avant de decider.

In [ ]:
ss.agg(F.min('ss_quantity'), F.max('ss_quantity'), F.min('ss_sales_price'), F.max('ss_sales_price'),
       F.min('ss_net_profit'), F.max('ss_net_profit')).show()
print('Quantite <= 0 :', ss.filter('ss_quantity <= 0').count())
print('Prix de vente = 0 :', ss.filter('ss_sales_price = 0').count())
print('Profit net < 0 :', ss.filter('ss_net_profit < 0').count())

## 6. store_returns : schema, volume, valeurs manquantes, doublons

In [ ]:
sr = charger_type(spark, 'store_returns')
sr.printSchema()
print('Lignes :', sr.count())
m = sr.agg(*[F.sum(F.col(c).isNull().cast('int')).alias(c) for c in noms('store_returns')]).first()
print({c: m[c] for c in noms('store_returns') if m[c]})
print('Repetitions de la cle', CLES['store_returns'], ':', sr.count() - sr.dropDuplicates(CLES['store_returns']).count())

## 7. Conversions de types

Une valeur non vide qui devient NULL apres conversion est comptee.

In [ ]:
print('store_sales   :', {c: v for c, v in echecs_conversion(spark, 'store_sales').items() if v} or 'aucun echec')
print('store_returns :', {c: v for c, v in echecs_conversion(spark, 'store_returns').items() if v} or 'aucun echec')
spark.stop()

## 8. Constats

Les constats chiffres sont ceux affiches ci-dessus ; leur interpretation et les regles de nettoyage associees sont dans le document des regles de nettoyage.